类别	核心工具 (示例)	最佳适用场景	优点	复杂度
轻量级缓存	Joblib, Cachew, Klepto	脚本、Jupyter Notebook 原型开发，希望用最少的代码实现函数级的持久化缓存。	学习曲线低，代码侵入性小，用装饰器即可。	低
工作流编排	Prefect, Dagster	构建长期运行的、复杂的生产级 ETL/ELT 数据流水线，需要任务调度、监控和健壮性。	功能强大，适合生产环境，提供UI和强大的依赖管理。	中到高
RAG 全栈框架	LlamaIndex, Haystack	专注于 RAG 应用开发，希望有模块化、可插拔的组件（加载器、分块器、检索器等），快速搭建原型或生产应用。	RAG 生态集成度高，简化了大量胶水代码，内置缓存和更新逻辑。	中

# 数据第一次清洗文件

In [ ]:
from pathlib import Path
import sys
import pickle
import numpy as np
import faiss
import pandas as pd
CACHE_DIR = "D:\WorkDirectory\PythonProject\RAG\cache"
import time
task_name = None
if task_name is None:
    task_name = str(time.time())+'_'
class CacheLoader:
    def __init__(self,cache_dir):
        self.cache_dir  = Path(cache_dir)
        # 检查路径
        if not self.cache_dir.exists():
            raise ImportError('文件路径不存在')
        self.chunks_path = self.cache_dir/'chunks.pkl'
        self.embeddings_path = self.cache_dir/'embeddings.npy'
        self.index_path = self.cache_dir/'index.faiss'

        self._chunks =None
        self._embeddings=None
        self._index = None
    '-----存在检查-----'
    def has_chunks(self):
        return self.chunks_path.exists()
    def has_embeddings(self):
        return self.embeddings_path.exists()
    def has_index(self):
        return self.index_path.exists()
    

    @property
    def chunks(self):
        if self._chunks is None and self.chunks_path.exists():
            with open(self.chunks_path,'rb')as f:
                self._chunks = pickle.load(f)
        return self._chunks


    @property
    def embeddings(self):
        if self._embeddings is None and self.embeddings_path.exists():
            self._embeddings = np.load(self.embeddings_path)
        return self._embeddings

    
    @property
    def index(self):
        if self._index is None and self.index_path.exists():
            self._index = faiss.read_index(str(self.index_path))
        return self._index

loader = CacheLoader(CACHE_DIR)

In [ ]:
# 读取文件
import pandas as pd 
import pickle
from pathlib import Path
# import os
# def filecheck(path):
#     os.path.split(path)
if not loader.has_chunks():
    # corpus_df = pd.read_parquet("D:\WorkDirectory\PythonProject\RAG\my_dataset\rag-bench-public-questions\data\train-00000-of-00001.parquet")
    corpus_df = pd.read_parquet(r"D:\WorkDirectory\PythonProject\RAG\my_dataset\wikidata\data\rag-mini-wikipedia\data\passages.parquet\part.0.parquet")
    print(corpus_df.head(5))
    # 文档分块
    from functions import FixedSizeChunkStrategy

    chunker  = FixedSizeChunkStrategy(chunk_size=500, overlap=50)
    # all_chunks内部需要展开，不能是列表嵌套列表（需要扁平化）
    all_chunks  = []
    for _,row in corpus_df.iterrows():
        chunks = chunker.chunk(row['passage'])
        all_chunks.extend(chunks)
    with open(loader.chunks_path,"wb") as f:
        pickle.dump(all_chunks,f)
    print(all_chunks)


                                              passage
id                                                   
0   Uruguay (official full name in  ; pron.  , Eas...
1   It is bordered by Brazil to the north, by Arge...
2   Montevideo was founded by the Spanish in the e...
3   The economy is largely based in agriculture (m...
4   According to Transparency International, Urugu...


In [ ]:
# 向量嵌入
# 准备一个文本嵌入向量模型
from sentence_transformers import SentenceTransformer
import os
os.environ['SENTENCE_TRANSFORMERS_HOME'] = '../cache/models/'
model_path = './cache/models'
if not loader.has_embeddings():
    if all_chunks is None:
        all_chunks = loader.chunks 
    # TODO 可以写成使用transformers的复杂做法，需要做池化提取句向量，以及批量编码
    if model_path:
        # model = SentenceTransformer('./cache/models')
        model = SentenceTransformer('./cache/models',local_files_only=True)
        model = SentenceTransformer('./cache/models',cache_folder='./cache/models')
    else:
        model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")
    # model.save_pretrained(<path>)
    model.save('./cache/models')

    # 将RAG分块嵌入成向量
    # 能直接全部编码，不需要自己实现
    embeddings = model.encode(all_chunks, batch_size=64, convert_to_tensor=False)
    embeddings = embeddings.astype('float32')
    np.save(loader.embeddings_path,embeddings)
    # TODO 优化1：调整batchsize
    # TODO 优化2：对chunk按长度预排序
    # TODO 优化3：启用FP16


In [ ]:
import faiss
import numpy as np
# 需求：embedding (float32类型数组)
if not loader.has_index():
    if embeddings is None:
        embeddings = loader.embeddings
    dimension = embeddings.shape[1]
    # --- 方案 A: 暴力检索 (高精度，小数据量首选) ---
    index = faiss.IndexFlatL2(dimension)  # L2 欧氏距离
    print(f"索引是否需要训练: {index.is_trained}") # Flat索引不需要训练

    # --- 方案 B: 快速近似检索 (大数据量，需要训练) ---
    # nlist = 100  # 聚类中心数量，通常设置为 sqrt(向量数量)
    # quantizer = faiss.IndexFlatL2(dimension)
    # index = faiss.IndexIVFFlat(quantizer, dimension, nlist)
    # # 注意: IVF索引在使用前必须经过训练
    # index.train(embeddings)

    # 3. 将向量添加到索引中
    index.add(embeddings)
    faiss.write_index(index)
    print(f"索引中向量总数: {index.ntotal}") # 应等于 embeddings 数量


向量维度: 1024, 向量数量: 4491
索引是否需要训练: True
索引中向量总数: 4491


In [6]:
query_df = pd.read_parquet(r'D:\WorkDirectory\PythonProject\RAG\my_dataset\wikidata\data\rag-mini-wikipedia\data\test.parquet\part.0.parquet')
print(query_df.head(5))
# USE_DATA_COUNT = query_df.shape[0]
USE_DATA_COUNT = 5
query_df = query_df.head(USE_DATA_COUNT)

                                             question     answer
id                                                              
0   Was Abraham Lincoln the sixteenth President of...        yes
2   Did Lincoln sign the National Banking Act of 1...        yes
4                    Did his mother die of pneumonia?         no
6       How many long was Lincoln's formal education?  18 months
8        When did Lincoln begin his political career?       1832


In [7]:
questions_list = []
answers_list = []
for _,row in query_df.iterrows():
    question = row['question']
    answer = row['answer']
    questions_list.append(question)
    answers_list.append(answer)

questions_embedding = model.encode(questions_list, batch_size=64, convert_to_tensor=False)
questions_embedding = questions_embedding.astype('float32')
# embeddings = model.encode(answers_list, batch_size=64, convert_to_tensor=False)


In [8]:
k = 5
distances, indices = index.search(questions_embedding, k)
# search功能：
# 对所有问题查询向量  [N个,D维]
# 遍历所有chunks的embedding
# 计算距离（创建方式决定，此处为L2距离）
# 从小到大排序，选出前K个

# 也就是，每个问题 -> K个向量[N,K,D]
# 存储简化成为 
# distances [N,K] 向量塌缩成一维的距离（此处为所有维度上的差值平方和）
# indices [N,K] 向量塌缩成一维的索引

# indice[0] 会先让N维度消失，每次得到一个[K个]{索引}（索引来自上面的分析）

# indices[0] 是检索到的片段在原始文档列表中的位置 如[0,1,2,3,4]
# distances[0] 是对应的距离（L2 越小越相似） 如 [1,3,4,8,9]

In [9]:
print(f"chunks 长度: {len(all_chunks)}")
print(f"索引中向量总数: {index.ntotal}")

chunks 长度: 4491
索引中向量总数: 4491


In [10]:
# 假设 chunks 是你分块后的原始文本列表，顺序与 embeddings 相同
# retrieved_chunks = [all_chunks[i] for i in indices[0]]

In [11]:
# 拼接检索到的内容作为上下文
# context = "\n\n".join(retrieved_chunks)
prompts_list = []
contexts_list = []
for i in range(len(questions_list)):
    retrieved_chunks = [all_chunks[j] for j in indices[i]]
    context = "\n\n".join(retrieved_chunks)
    contexts_list.append(context)
    prompt = f"""基于以下参考信息回答问题。
    参考信息：
    {context}
    只回答问题，不要生成新的问题或者回答问题以外的回答。
    回答结束时，输出eos标志
    问题：{questions_list[i]}
    回答："""
    prompts_list.append(prompt)
prompts_list


['基于以下参考信息回答问题。\n    参考信息：\n    Abraham Lincoln (February 12, 1809 â\x80\x93 April 15, 1865) was the sixteenth President of the United States, serving from March 4, 1861 until his assassination. As an outspoken opponent of the expansion of slavery in the United States, "[I]n his short autobiography written for the 1860 presidential campaign, Lincoln would describe his protest in the Illinois legislature as one that \'briefly defined his position on the slavery question, and so far as it goes, it was then the same that it is now." Thi\n\nOn November 6, 1860, Lincoln was elected as the 16th President of the United States, beating Democrat Stephen A. Douglas, John C. Breckinridge of the Southern Democrats, and John Bell of the new Constitutional Union Party. He was the first Republican president, winning entirely on the strength of his support in the North: he was not even on the ballot in nine states in the South, and won only 2 of 996 counties in the other Southern states. Lincoln gaine

In [12]:


import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
torch.cuda.empty_cache()
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,              # 启用4-bit量化[reference:1]
    bnb_4bit_quant_type="nf4",      # 使用nf4量化类型，质量更高[reference:2]
    bnb_4bit_compute_dtype=torch.float16, # 计算时使用float16，平衡速度与精度[reference:3]
    bnb_4bit_use_double_quant=True, # 启用双重量化，进一步压缩显存[reference:4]
)
# 加载模型和分词器 (以4-bit量化为例)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    quantization_config=bnb_config,
    device_map="auto"
)
responses_list = []
for i in range(len(prompts_list)):
    inputs = tokenizer(prompts_list[i],return_tensors="pt").to(model.device)
    response_ids = model.generate(inputs.input_ids,max_new_tokens=32,do_sample=False,stop_strings=["\n"],tokenizer=tokenizer,eos_token_id=tokenizer.eos_token_id)
    input_len = inputs.input_ids.shape[1]
    response_text = tokenizer.decode(response_ids[0][input_len:],skip_special_tokens=True)
    
    print(f"Prompt: {prompts_list[i]}{response_text}")
    responses_list.append(response_text)
    print("-" * 50)



Exception in thread Thread-6 (_readerthread):
Traceback (most recent call last):
  File "d:\Develop\miniconda\envs\ForResume\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "d:\Develop\miniconda\envs\ForResume\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "d:\Develop\miniconda\envs\ForResume\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "<frozen codecs>", line 322, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xb2 in position 7: invalid start byte


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Prompt: 基于以下参考信息回答问题。
    参考信息：
    Abraham Lincoln (February 12, 1809 â April 15, 1865) was the sixteenth President of the United States, serving from March 4, 1861 until his assassination. As an outspoken opponent of the expansion of slavery in the United States, "[I]n his short autobiography written for the 1860 presidential campaign, Lincoln would describe his protest in the Illinois legislature as one that 'briefly defined his position on the slavery question, and so far as it goes, it was then the same that it is now." Thi

On November 6, 1860, Lincoln was elected as the 16th President of the United States, beating Democrat Stephen A. Douglas, John C. Breckinridge of the Southern Democrats, and John Bell of the new Constitutional Union Party. He was the first Republican president, winning entirely on the strength of his support in the North: he was not even on the ballot in nine states in the South, and won only 2 of 996 counties in the other Southern states. Lincoln gained 1,8

In [ ]:
# 我现在有了参考内容、问题、回答、标准回答的矩阵
# 进行评估
import re
import json
def extract_json(text):
    match = re.search(r'\{.*?"factual_score".*?"completeness_score".*?"redundancy_score".*?\}', text, re.DOTALL)
    print("匹配结果",match)
    if match:
        return json.loads(match.group())
    return None
eval_scores = []
# for i in range(len(prompts_list)):
for i in range(1):
    eval_prompt = f"""你是一个RAG评估专家。请根据标准回答，评估生成回答的质量。

    问题：{questions_list[i]}
    标准回答：{answers_list[i]}
    检索到的上下文：{contexts_list[i]}
    生成回答：{responses_list[i]}

    请从以下维度打分（1-5分）：
    1. 事实一致性：生成回答的信息是否与标准回答一致？（5=完全一致，无矛盾）
    2. 完整性：生成回答是否覆盖了标准回答的所有关键点？（5=完全覆盖）
    3. 冗余性：生成回答是否包含多余或无关信息？（1=严重冗余，5=简洁相关）

    最终输出JSON格式：{{"factual_score": x, "completeness_score": y, "redundancy_score": z}}\n\n
    """
    eval_input = tokenizer(eval_prompt,return_tensors="pt").to(model.device)
    input_len = eval_input.input_ids.shape[0]
    eval_response_ids = model.generate(eval_input.input_ids,max_new_tokens=128,do_sample=False,eos_token_id = tokenizer.eos_token_id)
    eval_response_text = tokenizer.decode(eval_response_ids[0][input_len:],skip_special_tokens=True)
    print("评估结果",eval_response_text)
    try:
        eval_json = extract_json(eval_response_text)
    except Exception as e:
        # print(eval_response_text,"\n",e)
        eval_json = None
    eval_scores.append(eval_json)
eval_scores

你是一个RAG评估专家。请根据标准回答，评估生成回答的质量。

    问题：Was Abraham Lincoln the sixteenth President of the United States?
    标准回答：yes
    检索到的上下文：Abraham Lincoln (February 12, 1809 â April 15, 1865) was the sixteenth President of the United States, serving from March 4, 1861 until his assassination. As an outspoken opponent of the expansion of slavery in the United States, "[I]n his short autobiography written for the 1860 presidential campaign, Lincoln would describe his protest in the Illinois legislature as one that 'briefly defined his position on the slavery question, and so far as it goes, it was then the same that it is now." Thi

On November 6, 1860, Lincoln was elected as the 16th President of the United States, beating Democrat Stephen A. Douglas, John C. Breckinridge of the Southern Democrats, and John Bell of the new Constitutional Union Party. He was the first Republican president, winning entirely on the strength of his support in the North: he was not even on the ballot in nine states

[None, None, None, None, None]